# LLMSlim round-3 benchmark builds: `main`

This notebook builds everything except LongLLMLingua: LLMSlim (lexical and reranker), LLMLingua-2, retrieval baselines and the full contexts, on NaturalQuestions (2,255 questions), HotpotQA (2,000) and LoCoMo (1,540).

It **never calls a model API**. It only produces the exact context each system would send to the reader, so anyone can rerun it and compare the outputs byte for byte. The reader (Claude Haiku 5.5) and the judge run on these published contexts afterwards.

- **Code:** [Thanatos9404/llmslim](https://github.com/Thanatos9404/llmslim) at commit `05675495c8fad4672ef52fb53f9ac2d07fc0ba81`, `benchmarks/live/round3/`. Splits were committed before any result: `splits_r3.json`.
- **Data:** downloaded from each dataset's original source (lost-in-the-middle, HotpotQA, LoCoMo, LongMemEval) and checked by sha256.
- **Environments:** pinned Python 3.12 venvs (`kaggle_run.py setup`), separate from Kaggle's image.
- **Output:** `round3_main.zip`: one JSONL per system and suite (`id`, `context`, `tokens`, `build_ms`) and `manifest-main.json` (commit, GPUs, package versions, data and output hashes).

Settings: accelerator **GPU T4 x2**, internet **on**. Runtime: about 3-5 hours on a T4. A rerun resumes from the cases already built.

In [ ]:
import subprocess, sys
COMMIT = "05675495c8fad4672ef52fb53f9ac2d07fc0ba81"
JOB = "main"
R = "/tmp/src/benchmarks/live/round3/kaggle_run.py"
subprocess.run("rm -rf /tmp/src && git init -q /tmp/src && cd /tmp/src && git remote add origin https://github.com/Thanatos9404/llmslim && git fetch -q --depth 1 origin " + COMMIT + " && git checkout -q FETCH_HEAD && git log -1 --format='%H %s'", shell=True, check=True)
subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv", shell=True, check=True)
subprocess.run([sys.executable, R, "setup", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "data", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "build", "--work", "/tmp/r3", "--job", JOB], check=True)
subprocess.run([sys.executable, R, "package", "--work", "/tmp/r3", "--job", JOB, "--zip", "/kaggle/working/round3_" + JOB + ".zip"], check=True)